# P3 Execution Quality Dashboard
E1 Compression Breakout — V7.1 | Live signal tracking

Query `quant.db` → slippage, outcomes, edge-after-slippage, hard stop status.

In [ ]:
import sqlite3, os, warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
warnings.filterwarnings('ignore')

DB = os.path.expanduser('~/hummingbot/crypto-quant/data/quant.db')
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

df = pd.read_sql_query("""
    SELECT candidate_id, timestamp_utc, direction,
           decision_price, signal_level,
           fill_1m_price, fill_5m_price, fill_1h_price,
           slippage_1m_bps, slippage_5m_bps, slippage_1h_bps,
           slippage_1m_bucket,
           hyp_outcome, hyp_r_multiple, hyp_hold_minutes,
           hyp_same_candle_convention, would_have_won,
           p3_resolved, soft_filter_score, market_state
    FROM candidate_log
    WHERE disposition = 'SIGNAL_READY'
    ORDER BY timestamp_utc ASC
""", conn)

df['signal_dt'] = pd.to_datetime(df['timestamp_utc'], unit='ms', utc=True)
resolved = df[df['p3_resolved'] == 1].copy()
print(f"Total SIGNAL_READY: {len(df)} | Resolved: {len(resolved)} | Pending: {len(df)-len(resolved)}")
resolved.tail(5)[['signal_dt','direction','decision_price','fill_1m_price','slippage_1m_bps','slippage_1m_bucket','hyp_outcome','hyp_r_multiple']]

## Hard Stop Status

In [ ]:
# Hard stop thresholds (must match run_p3_resolve.py)
STOP_AVG_SLIP  = 15.0
STOP_DANGER_RT = 0.30
BACKTEST_WR    = 0.594
BACKTEST_AVG_R = 0.20
SL_PCT         = 0.015

if len(resolved) == 0:
    print("No resolved candidates yet.")
else:
    avg_slip   = resolved['slippage_1m_bps'].mean()
    danger_rt  = (resolved['slippage_1m_bucket'] == 'danger').mean()
    hyp_wr     = resolved['would_have_won'].mean()
    avg_r      = resolved['hyp_r_multiple'].mean()
    slip_as_r  = (avg_slip / 10000) / SL_PCT
    edge       = BACKTEST_AVG_R - slip_as_r

    stops = []
    if avg_slip > STOP_AVG_SLIP:   stops.append(f"AVG SLIPPAGE {avg_slip:.1f} bps > {STOP_AVG_SLIP}")
    if danger_rt > STOP_DANGER_RT: stops.append(f"DANGER RATE {danger_rt:.1%} > {STOP_DANGER_RT:.0%}")
    if edge < 0:                   stops.append(f"EDGE NEGATIVE: {edge:+.3f}R")

    print(f"N resolved:          {len(resolved)}")
    print(f"Avg slippage (1m):   {avg_slip:+.2f} bps  (hard stop: >{STOP_AVG_SLIP})")
    print(f"Danger rate:         {danger_rt:.1%}  (hard stop: >{STOP_DANGER_RT:.0%})")
    print(f"Hyp win rate:        {hyp_wr:.1%}  (backtest ref: {BACKTEST_WR:.1%})")
    print(f"Avg R-multiple:      {avg_r:+.3f}")
    print(f"Edge after slippage: {edge:+.3f}R  ({'POSITIVE' if edge>0 else 'NEGATIVE'})")
    print()
    if stops:
        print("HARD STOPS TRIGGERED:")
        for s in stops: print(f"  {s}")
    else:
        note = "  (n<20 — not yet decision-grade)" if len(resolved)<20 else ""
        print(f"All hard stops clear{note}")

## Slippage Distribution

In [ ]:
if len(resolved) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    for col, label, color in [
        ('slippage_1m_bps','1m (primary)','steelblue'),
        ('slippage_5m_bps','5m (secondary)','orange'),
        ('slippage_1h_bps','1h (stress)','gray'),
    ]:
        vals = resolved[col].dropna()
        if len(vals): axes[0].hist(vals, bins=15, alpha=0.6, label=label, color=color)
    axes[0].axvline(0, color='black', linestyle='--', linewidth=0.8)
    axes[0].axvline(10, color='orange', linestyle=':', linewidth=1, label='borderline')
    axes[0].axvline(20, color='red',    linestyle=':', linewidth=1, label='danger')
    axes[0].axvline(15, color='red',    linestyle='--', linewidth=1.5, label='hard stop avg')
    axes[0].set_title('Slippage Distribution (bps)')
    axes[0].set_xlabel('Slippage bps'); axes[0].legend(fontsize=7)

    buckets = resolved['slippage_1m_bucket'].value_counts().reindex(['safe','borderline','danger'], fill_value=0)
    axes[1].bar(buckets.index, buckets.values, color=['#4CAF50','#FF9800','#F44336'])
    axes[1].set_title('Slippage Bucket (1m model)')
    for i, v in enumerate(buckets.values):
        axes[1].text(i, v+0.1, str(v), ha='center', fontweight='bold')

    r_sorted = resolved.sort_values('signal_dt')
    axes[2].plot(range(len(r_sorted)), r_sorted['slippage_1m_bps'], 'o-', color='steelblue', markersize=5)
    axes[2].axhline(15, color='red', linestyle='--', linewidth=1, label='hard stop')
    axes[2].axhline(0,  color='black', linestyle='-', linewidth=0.5)
    axes[2].set_title('Slippage Over Time (1m)')
    axes[2].set_xlabel('Signal #'); axes[2].set_ylabel('bps'); axes[2].legend(fontsize=8)

    plt.tight_layout(); plt.show()

## Hypothetical Outcomes

In [ ]:
if len(resolved) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    outcomes = resolved['hyp_outcome'].value_counts()
    colors_map = {'TP':'#4CAF50','SL':'#F44336','TIME':'#9E9E9E'}
    axes[0].pie(outcomes.values, labels=outcomes.index,
                colors=[colors_map.get(k,'gray') for k in outcomes.index],
                autopct='%1.0f%%', startangle=90)
    axes[0].set_title(f'Outcomes  (n={len(resolved)})')

    r_vals = resolved['hyp_r_multiple'].dropna()
    axes[1].hist(r_vals, bins=20, color='steelblue', edgecolor='white')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[1].axvline(r_vals.mean(), color='red', linestyle='--', linewidth=1.5,
                    label=f'mean {r_vals.mean():.2f}R')
    axes[1].set_title('R-Multiple Distribution'); axes[1].legend()

    r_sorted = resolved.sort_values('signal_dt')['hyp_r_multiple'].dropna()
    axes[2].plot(range(len(r_sorted)), r_sorted.cumsum(), 'o-', color='steelblue', markersize=4)
    axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[2].set_title('Cumulative R'); axes[2].set_xlabel('Signal #'); axes[2].set_ylabel('R')

    plt.tight_layout(); plt.show()

## Fill Model Comparison + Hold Time

In [ ]:
if len(resolved) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    model_avgs = {
        '1m (primary)':   resolved['slippage_1m_bps'].mean(),
        '5m (secondary)': resolved['slippage_5m_bps'].mean(),
        '1h (stress)':    resolved['slippage_1h_bps'].mean(),
    }
    bars = axes[0].bar(list(model_avgs.keys()), list(model_avgs.values()),
                       color=['steelblue','orange','gray'])
    axes[0].axhline(15, color='red', linestyle='--', linewidth=1.5, label='hard stop')
    axes[0].axhline(0,  color='black', linewidth=0.5)
    axes[0].set_title('Avg Slippage by Fill Model'); axes[0].set_ylabel('bps'); axes[0].legend()
    for bar, v in zip(bars, model_avgs.values()):
        if not np.isnan(v):
            axes[0].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=9)

    hold = resolved['hyp_hold_minutes'].dropna()
    axes[1].hist(hold, bins=20, color='steelblue', edgecolor='white')
    axes[1].axvline(hold.mean(), color='red', linestyle='--',
                    linewidth=1.5, label=f'mean {hold.mean():.0f}min')
    axes[1].set_title('Hold Time Distribution'); axes[1].set_xlabel('Minutes'); axes[1].legend()

    plt.tight_layout(); plt.show()

    print("\nFill model comparison:")
    for k, v in model_avgs.items():
        print(f"  {k:<20} {v:+.2f} bps")

## Signal Detail Table

In [ ]:
if len(resolved) > 0:
    pd.set_option('display.max_rows', 50)
    pd.set_option('display.float_format', '{:.2f}'.format)
    display_cols = [
        'signal_dt','direction','decision_price','fill_1m_price',
        'slippage_1m_bps','slippage_1m_bucket',
        'hyp_outcome','hyp_r_multiple','hyp_hold_minutes',
        'hyp_same_candle_convention','soft_filter_score'
    ]
    resolved.sort_values('signal_dt')[display_cols].reset_index(drop=True)

## Edge After Slippage

In [ ]:
if len(resolved) > 0:
    avg_slip  = resolved['slippage_1m_bps'].mean()
    slip_as_r = (avg_slip / 10000) / SL_PCT
    edge      = BACKTEST_AVG_R - slip_as_r

    print('='*50)
    print('EDGE AFTER SLIPPAGE')
    print('='*50)
    print(f'Backtest avg R:      {BACKTEST_AVG_R:+.3f}R')
    print(f'Avg slippage (1m):   {avg_slip:+.2f} bps')
    print(f'Slippage cost:      -{slip_as_r:.3f}R')
    print(f'Edge after slippage: {edge:+.3f}R')
    print()
    if edge > 0:
        print('POSITIVE — executable from slippage standpoint')
    else:
        print('NEGATIVE — investigate before going live')
    print(f'\nn={len(resolved)} signals. Need 20+ for conclusions.')